# H02 · DMA double buffering, battery life and aliasing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/08_hardware_electronics/H02_dma_double_buffering_battery_life_and_aliasing.ipynb)

**Track:** Hardware and electronics · **Skill axis (sheet):** Hardware and electronics · **Time:** about 50 min

**Skill Hub concepts:** DMA double buffering; Duty-cycled average current; Nyquist sampling and aliasing

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("H02")

## Map: three budgets every sensor node has

**DMA double buffering (ping-pong):** the ADC fills a circular buffer of $n$ samples by DMA. The half-transfer
interrupt fires when the first half is full, the transfer-complete interrupt when the second half is full.
```
 buffer:  [ half 0 ............ | half 1 ............ ]
 DMA writing half 1  ->  CPU processes half 0    (and vice versa)
 deadline per half = (n/2) / f_s ; processing must finish before DMA wraps into the half you are reading
```

**Duty-cycled average current:** $\bar I = \dfrac{\sum_k I_k t_k + I_{sleep}\,(T - \sum_k t_k)}{T}$, battery life
$= \dfrac{C \times \text{derating}}{\bar I}$ (derate about 0.7 to 0.85 for temperature, self-discharge, cutoff voltage).

**Nyquist:** sample at $f_s > 2 f_{max}$. Anything above $f_s/2$ folds back:
$f_{alias} = \lvert f - f_s \cdot \mathrm{round}(f / f_s)\rvert$. Only an analog anti-alias filter before the ADC
prevents it; filtering after sampling is too late.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Task 1: ping-pong buffer rules

In [ ]:
def half_deadline_s(fs, n):
    raise NotImplementedError  # TODO

def half_to_process(dma_index, n):
    """The DMA is about to write buffer[dma_index]. Which half (0 or 1) is safe for the CPU to read?"""
    raise NotImplementedError  # TODO

lab.check("pingpong", half_deadline_s, half_to_process)

In [ ]:
%%writefile h02_pingpong.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/h02.h
// Grade it with lab.check("pingpong", "h02_pingpong.c"). For C++, name the file .cpp in both places.
#include "h02.h"

double half_deadline_s(double fs, size_t n) { (void)fs; (void)n; return 0; /* TODO */ }
int half_to_process(size_t dma_index, size_t n) { (void)dma_index; (void)n; return 0; /* TODO */ }

In [ ]:
# lab.check("pingpong", "h02_pingpong.c")  # uncomment after writing the C/C++ cell above

## Task 2: battery life of a duty-cycled sensor
`phases` = list of `(current_mA, seconds)` per wake-up; one wake-up every `period_s`; asleep otherwise.

In [ ]:
def average_current_mA(phases, sleep_mA, period_s):
    raise NotImplementedError  # TODO

def battery_life_hours(capacity_mAh, avg_mA, derating=0.8):
    raise NotImplementedError  # TODO

lab.check("battery", average_current_mA, battery_life_hours)

In [ ]:
%%writefile h02_battery.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/h02.h
// Grade it with lab.check("battery", "h02_battery.c"). For C++, name the file .cpp in both places.
#include "h02.h"

double average_current_mA(const double *current_mA, const double *duration_s, size_t n_phases, double sleep_mA, double period_s) {
    (void)current_mA; (void)duration_s; (void)n_phases; (void)sleep_mA; (void)period_s;
    return 0; /* TODO: charge per period / period */
}

double battery_life_hours(double capacity_mAh, double avg_mA, double derating) { (void)capacity_mAh; (void)avg_mA; (void)derating; return 0; /* TODO */ }

In [ ]:
# lab.check("battery", "h02_battery.c")  # uncomment after writing the C/C++ cell above

## Task 3: where does a frequency land after sampling?

In [ ]:
def alias(f, fs):
    raise NotImplementedError  # TODO

lab.check("alias", alias)

In [ ]:
%%writefile h02_alias.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/h02.h
// Grade it with lab.check("alias", "h02_alias.c"). For C++, name the file .cpp in both places.
#include "h02.h"

double alias(double f, double fs) { (void)f; (void)fs; return 0; /* TODO: distance to the nearest multiple of fs */ }

In [ ]:
# lab.check("alias", "h02_alias.c")  # uncomment after writing the C/C++ cell above

In [ ]:
fs, t_fine = 1000.0, np.linspace(0, 0.02, 2000)
t_s = np.arange(0, 0.02, 1 / fs)
f = 900.0
plt.plot(t_fine * 1e3, np.sin(2 * np.pi * f * t_fine), alpha=0.4, label="900 Hz signal")
plt.plot(t_s * 1e3, np.sin(2 * np.pi * f * t_s), "o-", label="sampled at 1 kHz")
plt.xlabel("ms"); plt.legend(); plt.title("the samples trace a 100 Hz wave"); plt.show()

## Checkpoint: Math
**M1 (Apply).** 2048-sample circular buffer at 48 kHz. Processing deadline per half, in milliseconds (2 decimals)?

**M2 (Analyze).** A sensor wakes every 5 minutes: 5 mA for 50 ms (read), 10 mA for 5 ms (BLE), 2 uA asleep, on a
220 mAh coin cell with 0.8 derating. Battery life in hours?

**M3 (Apply).** A 1.2 kHz vibration sampled at 1 kHz shows up at what frequency?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Apply).** Motor current is sampled at 10 kHz but the PWM runs at 25 kHz, and the logged current shows a slow
5 kHz ripple that is not real. Best fix?
- a) a digital low-pass filter on the logged data
- b) sample synchronously with the PWM (trigger the ADC from the timer at the period center) and/or add an analog anti-alias filter
- c) increase the log rate

**K2 (Evaluate).** In the battery example, which phase dominates the average, and what would you change first to
double the battery life?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `h02.h`).
3. In `roboprep/checks/h02.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()